# 1. 工具的调用方式
## 1.1 直接调用

In [1]:
from langchain_core.tools import tool

@tool
def get_weather(city: str) -> str:
    """
    获取指定城市的天气信息
    参数:
    city: 城市名称，如"北京"、"上海"
    返回:
    天气信息字符串
    """
    # 你的实现
    return city + "晴天，温度 15°C"


In [2]:
# 使用 .invoke() 方法（如果上面的 get_weather 是个函数，那就没有 invoke 方法，但是作为工具就有）
result = get_weather.invoke({"city": "北京"})
print(result)

北京晴天，温度 15°C


## 1.2. 基于模型调用

In [3]:
# 模型初始化
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

load_dotenv(override=True)

model = init_chat_model(
    model="deepseek-flash",
    model_provider="openai",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL")
)


In [4]:
from langchain_core.tools import tool

# 定义工具 1. 工具的调用方式：不使用 @tool 修饰
@tool
def get_weather(city: str) -> str:
    """获取指定城市的天气"""
    # 你的实现
    return "晴天，温度 15°C"

# 绑定工具
model_with_tools = model.bind_tools([get_weather])

# AI 可以决定是否调用工具
response = model_with_tools.invoke("北京天气如何？")
# response = model_with_tools.invoke("2 + 3 = ？")

# 检查 AI 是否要调用工具
if response.tool_calls:
    print("AI 想调用工具：", response.tool_calls)
else:
    print("AI 直接回答：", response.content)

AI 想调用工具： [{'name': 'get_weather', 'args': {'city': '北京'}, 'id': 'call_00_p0mK45i9hezgLYW6WlRE4295', 'type': 'tool_call'}]


# 2. 从 Message 流转看工具的调用

In [ ]:
# 模型初始化
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

load_dotenv(override=True)

model = init_chat_model(
    model="deepseek-flash",
    model_provider="openai",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL")
)


使用 @tool 修饰

In [5]:
from langchain.messages import HumanMessage, ToolMessage
from rich import print as rprint

@tool(description="根据城市名称查询当日天气的工具")
def get_weather(city: str):
    """
    天气查询工具
    """
    return f"{city}天气晴朗"

# 将模型和工具绑定
tools = model.bind_tools([get_weather])

messages = [HumanMessage("今天北京天气如何？")]

# 模型生成调用工具请求
response = tools.invoke(messages)

# 添加 AIMessage
messages.append(response)

rprint(response)
"""""
tool_calls=[
    {
        'name': 'get_weather',
        'args': {'city': '北京'},
        'id': 'call_00_pBXhcQMictOgXHzBbGvx0463',
        'type': 'tool_call'
    }
]
"""""
tool_calls = response.tool_calls

for tool_call in tool_calls:
    if tool_call["name"] == "get_weather":
        # 返回的是 ToolMessage 类型消息，添加到消息列表中
        tool_response = get_weather.invoke(tool_call)
        print(type(tool_response))
        messages.append(tool_response)

print("=====================> messages <=====================")

for msg in messages:
    msg.pretty_print()
print("=====================> messages <=====================")

final_response = model_with_tools.invoke(messages)
print(f"final_response: \n{final_response}")

AIMessage(
    content='',
    additional_kwargs={'refusal': None},
    response_metadata={
        'token_usage': {
            'completion_tokens': 54,
            'prompt_tokens': 295,
            'total_tokens': 349,
            'completion_tokens_details': {
                'accepted_prediction_tokens': None,
                'audio_tokens': None,
                'reasoning_tokens': 15,
                'rejected_prediction_tokens': None
            },
            'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0},
            'prompt_cache_hit_tokens': 0,
            'prompt_cache_miss_tokens': 295
        },
        'model_provider': 'openai',
        'model_name': 'deepseek-flash',
        'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6',
        'id': 'da54ba40-7826-4d1d-94f6-8a3016ceadd6',
        'finish_reason': 'tool_calls',
        'logprobs': None
    },
    id='lc_run--01a1ba35-ab5f-7654-8039-36d8517c1eb5-0',
    tool_calls=[
        {
            'name': 'get_weather',
            'args': {'city': '北京'},
            'id': 'call_00_pBXhcQMictOgXHzBbGvx0463',
            'type': 'tool_call'
        }
    ],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 295,
        'output_tokens': 54,
        'total_tokens': 349,
        'input_token_details': {'cache_read': 0},
        'output_token_details': {'reasoning': 15}
    }
)

# 3. 工具描述的各部分详解
## 3.1 了解 convert_to_openai_tool
执行 model.bind_tools([get_weather]) ，底层最终会调用 convert_to_openai_tool 生成工具描述。所以我们可以直接调用后者查看解析后的工具描述。

In [ ]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint

def get_weather(city: str):
    return f"{city}天气晴朗"

rprint(convert_to_openai_tool(get_weather))

## 3.2 description 说明

In [ ]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint

def get_weather(city: str):
    """
    天气查询工具
    """
    return f"{city}天气晴朗"

rprint(convert_to_openai_tool(get_weather))

## 3.2 参数说明
使用 Args: 、 Returns: 、 Raises: 等关键字，这种方式可读性强。Agent通过工具的这些注释来理解
工具的用途和调用时机，因此清晰、准确的文档字符串是工具能被正确调用的前提

In [ ]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint

# 使用 Args: 、 Returns: 、 Raises: 等关键字，这种方式可读性强。Agent通过工具的这些注释来理解
# 工具的用途和调用时机，因此清晰、准确的文档字符串是工具能被正确调用的前提
def get_weather(city: str):
    """
    天气查询工具

    Args:
        city: 城市名称
    """
    return f"{city}天气晴朗"

rprint(convert_to_openai_tool(get_weather))

## 3.2 参数类型说明
参数类型来源于函数的类型注解。要求：如果在 docstring 中声明了参数的描述，则必须在函数声明处指明参数的类型，否则会报错

下面描述中有 city，函数声明处必须有类型
def get_weather(city: str="北京"):
    """
    天气查询工具

    Args:
        city: 城市名称
    """
    return f"{city}天气晴朗"

In [ ]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint

def get_weather(city: str):
    """
    天气查询工具
    """
    return f"{city}天气晴朗"

rprint(convert_to_openai_tool(get_weather))

## 3.3 参数默认值说明
一旦参数设置了默认值，则打印的结果中的 required 字段中就不再包含此参数（不再是必传）

In [ ]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint

def get_weather(city: str="北京"):
    """
    天气查询工具

    Args:
        city: 城市名称
    """
    return f"{city}天气晴朗"

rprint(convert_to_openai_tool(get_weather))